# 09 — Pseudobulk: which genes change with age, with mice as samples?

The classifier weights in 05b are interpretation, not a test. Here each tissue × cell type is summed per mouse (**pseudobulk**) and tested with DESeq2, so the sample size is the number of mice, as it should be (Squair et al. 2021). Then three questions:

1. How many genes change in each row, and are robust rows different from batch-driven ones?
2. **Shared or private aging?** Do rows change the same genes in the same direction, beyond what shared mice alone would produce?
3. **Do the atlas's published "global aging genes" hold up?** Zhang et al. 2021 (eLife) reported 330 genes that change with age across many cell types, tested with cells as samples and no batch check.

Reads `figure/atlas/atlas.csv` (07) and `data/external/global_aging_genes.tsv`.

### 1. Setup

In [ ]:
import sys
sys.path.append("..")          # so "src" can be imported from the notebook folder

import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import false_discovery_control, spearmanr
from src import aging as ag

A = ag.paths("atlas")
atlas = pd.read_csv(A.figure("atlas.csv"))
atlas["robust"] = atlas["verdict"].eq("age signal") & ~atlas["tech_confounded"].astype(bool)

from scipy.stats import binomtest

### 2. DESeq2 per row

Every testable, non-void row; mice with ≥ 20 cells of the type; design `~group` (old vs. young). Genes are annotated afterwards (foreign to the cell type, dissociation, circadian), not removed, so the counts below can be read with and without them. About 5 s per row.

In [ ]:
use = atlas[~atlas["verdict"].isin(["void", "not testable"])]
de, effects, metas = {}, {}, {}
for tissue in sorted(use["tissue"].unique()):
    a = sc.read_h5ad(ag.paths(tissue).processed)
    ct_means = ag.celltype_means(a)
    for ct in use.loc[use["tissue"] == tissue, "cell_type"]:
        counts, meta = ag.pseudobulk(a, ct, **ag.main_kw(ct, tissue))
        groups = meta["group"].value_counts().reindex(["old", "young"], fill_value=0)
        if groups.min() < 2:          # a group can vanish once mice with < 20 cells are dropped
            print("  skipped", ct, dict(groups))
            continue
        r = ag.deseq(counts, meta)
        r["foreign"] = r.index.isin(ag.foreign_genes(ct_means, ct))
        r["flag"] = np.select([r.index.isin(ag.DISSOCIATION), r.index.isin(ag.CIRCADIAN)],
                              ["dissociation", "circadian"], "")
        de[(tissue, ct)] = r
        effects[(tissue, ct)] = ag.log_cpm_change(counts, meta)
        metas[(tissue, ct)] = meta
    print(tissue, "done")
    del a
pd.concat({f"{t} | {c}": r for (t, c), r in de.items()}, names=["row", "gene"]).round(5) \
  .to_csv(A.figure("pseudobulk_de.csv.gz"))

### 3. How many genes change per row

In [ ]:
n = pd.DataFrame([{"tissue": t, "cell_type": c, "n_mice": len(metas[(t, c)]),
                   "de_all": (r["padj"] < 0.1).sum(),
                   "de_own": ((r["padj"] < 0.1) & ~r["foreign"] & (r["flag"] == "")).sum(),
                   "up_own": ((r["padj"] < 0.1) & ~r["foreign"] & (r["flag"] == "") & (r["log2FoldChange"] > 0)).sum()}
                  for (t, c), r in de.items()]).merge(
    atlas[["tissue", "cell_type", "compartment", "verdict", "robust"]], on=["tissue", "cell_type"])
n.to_csv(A.figure("pseudobulk_counts.csv"), index=False)
print(n.groupby("robust")[["de_own"]].describe().round(1))
n.sort_values("de_own", ascending=False)

### 4. Shared or private aging: do rows agree?

For every pair of rows, the Spearman correlation of their per-gene effects (mean log2 CPM old − young), over genes expressed in both. **Rows share mice**, and a mouse-level batch would make rows agree for the wrong reason. So the null is a **mouse-consistent permutation**: the old/young labels of the 23 mice are shuffled once and applied to every tissue alike, which keeps the shared-mouse structure but removes age. A pair agrees beyond that only if its real correlation is above the 95th percentile of 200 permutations.

In [ ]:
keys = list(effects)
expressed = {k: de[k].index[de[k]["baseMean"] > 20] for k in keys}

def corr_matrix(eff):
    M = pd.DataFrame(np.nan, index=range(len(keys)), columns=range(len(keys)))
    for i in range(len(keys)):
        for j in range(i + 1, len(keys)):
            g = expressed[keys[i]].intersection(expressed[keys[j]])
            if len(g) >= 200 and eff[keys[i]] is not None and eff[keys[j]] is not None:
                M.iloc[i, j] = M.iloc[j, i] = spearmanr(eff[keys[i]][g], eff[keys[j]][g])[0]
    return M

real = corr_matrix(effects)

# mouse-consistent permutation null; needs the per-mouse counts again
pb_cache = {}
for tissue in sorted({t for t, _ in keys}):
    a = sc.read_h5ad(ag.paths(tissue).processed)
    for (t, c) in [k for k in keys if k[0] == tissue]:
        pb_cache[(t, c)] = ag.pseudobulk(a, c, **ag.main_kw(c, tissue))
    del a
mouse_group = pd.concat([m["group"] for _, m in pb_cache.values()]).groupby(level=0).first()

rng = np.random.default_rng(0)
null = []
for _ in range(200):
    perm = pd.Series(rng.permutation(mouse_group.to_numpy()), index=mouse_group.index)
    eff = {}
    for k, (counts, meta) in pb_cache.items():
        m = meta.assign(group=perm.loc[meta.index].to_numpy())
        eff[k] = ag.log_cpm_change(counts, m) if m["group"].nunique() == 2 else None
    null.append(corr_matrix(eff).to_numpy())
null = np.stack(null)

In [ ]:
q95 = np.nanquantile(null, 0.95, axis=0)
labels = [f"{t}: {c[:30]}" for t, c in keys]
real.index = real.columns = labels
beyond = pd.DataFrame(real.to_numpy() > q95, index=labels, columns=labels)
pairs = []
for i in range(len(keys)):
    for j in range(i + 1, len(keys)):
        pairs.append({"row_a": labels[i], "row_b": labels[j], "rho": real.iloc[i, j],
                      "null_q95": q95[i, j], "beyond_null": bool(beyond.iloc[i, j]),
                      "same_tissue": keys[i][0] == keys[j][0],
                      "comp_a": ag.compartment(keys[i][1]), "comp_b": ag.compartment(keys[j][1])})
pairs = pd.DataFrame(pairs).dropna(subset=["rho"])
pairs.round(3).to_csv(A.figure("pseudobulk_pair_agreement.csv"), index=False)
print("pairs:", len(pairs), "| beyond the shared-mouse null:", pairs["beyond_null"].sum())
print(pairs.groupby("same_tissue")[["rho", "null_q95"]].median().round(3))
pairs.groupby(["comp_a", "comp_b"])["beyond_null"].agg(["size", "sum", "mean"]).round(2)

In [ ]:
robust_keys = [labels[i] for i, k in enumerate(keys)
               if atlas.set_index(["tissue", "cell_type"]).loc[k, "robust"]]
sub = real.loc[robust_keys, robust_keys]
g = sns.clustermap(sub.fillna(0), cmap="RdBu_r", center=0, vmin=-0.5, vmax=0.5,
                   figsize=(0.3 * len(sub) + 4, 0.3 * len(sub) + 4), xticklabels=True, yticklabels=True)
g.ax_heatmap.tick_params(labelsize=6)
g.savefig(A.figure("pseudobulk_agreement_robust.png"), dpi=200, bbox_inches="tight")

### 5. Genes shared across robust rows

A gene that rises (or falls) significantly in several robust rows, from different tissues, with no foreign, dissociation or circadian flag. Counted in **tissues**, not rows, so several cell types of one organ (e.g. kidney tubule segments) count once.

In [ ]:
robust_rows = [k for k in keys if atlas.set_index(["tissue", "cell_type"]).loc[k, "robust"]]
hits = []
for (t, c) in robust_rows:
    r = de[(t, c)]
    r = r[(r["padj"] < 0.1) & ~r["foreign"] & (r["flag"] == "")]
    hits.append(pd.DataFrame({"gene": r.index, "dir": np.sign(r["log2FoldChange"]), "tissue": t, "cell_type": c}))
hits = pd.concat(hits)
shared = (hits.groupby(["gene", "dir"]).agg(tissues=("tissue", "nunique"), rows=("cell_type", "size"),
                                             where=("tissue", lambda v: ", ".join(sorted(set(v)))))
          .reset_index().sort_values(["tissues", "rows"], ascending=False))
shared["direction"] = shared["dir"].map({1.0: "up in old", -1.0: "down in old"})
shared.drop(columns="dir").to_csv(A.figure("pseudobulk_shared_genes.csv"), index=False)
shared[shared["tissues"] >= 3].drop(columns="dir").head(40)

### 6. Do the published global aging genes hold up?

Zhang et al. 2021 list 330 genes; 283 have a direction (`gag_score_coef` > 0: up with age, < 0: down). The other 47 have a coefficient of 0 and no consistent direction even in their data (up in 22–80% of cell types), so they are left out of every direction test. For every row, among those genes expressed in it: the fraction whose pseudobulk log2 fold change has the **published sign**, and the fraction that is also significant (padj < 0.1) in that direction. A fraction near 0.5 means agreement no better than a coin. Rows are split into robust and other, and the genes into those on the dissociation list and the rest.

In [ ]:
gag = pd.read_csv(ag.ROOT / "data" / "external" / "global_aging_genes.tsv", sep="\t")
gag["gene"] = gag["global_aging_genes"].map(lambda g: ag.GENE_ALIASES.get(g, g))
gag["pub_dir"] = np.sign(gag["gag_score_coef"])
print((gag["pub_dir"] == 0).sum(), "genes without a published direction are left out")
gag = gag[gag["pub_dir"] != 0].reset_index(drop=True)
print(len(gag), "genes;", (gag["pub_dir"] > 0).sum(), "up,", (gag["pub_dir"] < 0).sum(), "down;",
      gag["gene"].isin(ag.DISSOCIATION).sum(), "on the dissociation list:",
      sorted(gag.loc[gag["gene"].isin(ag.DISSOCIATION), "gene"]))

rep = []
for (t, c), r in de.items():
    x = r.reindex(gag["gene"]).assign(pub_dir=gag["pub_dir"].to_numpy()).dropna(subset=["log2FoldChange"])
    x = x[x["baseMean"] > 20]
    agree = np.sign(x["log2FoldChange"]) == x["pub_dir"]
    rep.append({"tissue": t, "cell_type": c, "genes": len(x), "sign_agree": agree.mean(),
                "sig_agree": (agree & (x["padj"] < 0.1)).mean(),
                "sign_agree_no_dissociation": agree[~x.index.isin(ag.DISSOCIATION)].mean(),
                "binom_p": binomtest(int(agree.sum()), len(x), 0.5).pvalue if len(x) else np.nan})
rep = pd.DataFrame(rep).merge(atlas[["tissue", "cell_type", "compartment", "verdict", "robust"]],
                              on=["tissue", "cell_type"])
rep.round(3).to_csv(A.figure("global_aging_genes_replication.csv"), index=False)
rep.groupby("robust")[["sign_agree", "sig_agree", "sign_agree_no_dissociation"]].describe().round(2).T

In [ ]:
rep.sort_values("sign_agree", ascending=False)[["tissue", "cell_type", "compartment", "robust", "genes",
                                                 "sign_agree", "sig_agree", "binom_p"]].round(3)

### 7. Which published genes replicate most widely?

In [ ]:
per_gene = []
for g, d in gag.set_index("gene").iterrows():
    s = [np.sign(r.loc[g, "log2FoldChange"]) == d["pub_dir"] for k, r in de.items()
         if k in robust_rows and g in r.index and r.loc[g, "baseMean"] > 20]
    if s:
        per_gene.append({"gene": g, "published": "up" if d["pub_dir"] > 0 else "down",
                         "robust_rows": len(s), "agree": float(np.mean(s)),
                         "flag": "dissociation" if g in ag.DISSOCIATION else ""})
per_gene = pd.DataFrame(per_gene).sort_values(["agree", "robust_rows"], ascending=False)
per_gene.to_csv(A.figure("global_aging_genes_per_gene.csv"), index=False)
print(per_gene["agree"].describe().round(2))
per_gene.head(25)

### Findings

DESeq2 on 55 tissue × cell type rows (22 robust), mice as samples, design old vs. young. Gene counts below exclude foreign, dissociation and circadian genes.

- **Robust rows change more genes:** median 175 genes at padj < 0.1 against 58 in the other rows. Lung fibroblast confirms the 05b weights with a real test: *Cdkn1a* log2FC +2.5, *Cfb* +1.4, *Col1a1* −1.4, *Col3a1* −1.8 (all padj < 0.01).
- **Aging is mostly private to each cell type.** Of 1,485 row pairs, 116 (7.8%) agree beyond the shared-mouse null, against 5% expected by chance. Between tissues the median correlation of effects is 0.06, under a null 95th percentile of 0.22. Within one tissue rows correlate more (0.28), but the shared-mouse null is higher still (0.39): **agreement inside an organ is what shared mice and batch produce on their own.**
- **A small shared core, across 4–5 tissues of robust rows:** MHC class I up (*H2-D1*, *H2-K1*, *B2m*), *Cdkn1a* up (5 tissues; see 10 for why it is not called an aging marker), inflammation and interferon up (*Tnf*, *Casp1*, *Casp4*, *Gbp3*), *S100a6*, *S100a11*, *Lgals3* up; *Tsc22d3* (GILZ) down in 5 tissues — a glucocorticoid-responsive gene, so possibly time-of-day or stress hormone rather than aging. *Cdkn2a* (p16) rises in 6 rows across 4 tissues once counts are summed per mouse (log2FC 1.8–5.2, low counts), though it is detected in under 1% of single cells; Kimmel's release lacks the gene, so it can't be replicated.
- **The published global aging genes split in two.** Of Zhang et al. 2021's 330 genes, 283 have a direction (47 have none even in their own data). Per row, sign agreement is a coin flip (robust 0.50, others 0.54). But the **up** genes replicate — 60 of 93 agree in most robust rows (binomial p = 0.007), led by MHC class I and II (*H2-D1*, *H2-K1* 22 of 22 rows, *B2m* 21 of 22, *Cd74*, *H2-Aa*, *H2-Ab1*) — while the **down** genes do not, and lean the other way: 68 of 190 (p = 0.0001), worst for ribosomal genes (30% agreement). A cell-as-sample test with no batch control can't separate a falling ribosomal fraction from a processing difference; with mice as samples, that half of the list does not hold.
- **Read as:** the antigen-presentation / inflammation axis (MHC up) is the one aging signal shared across cell types, tissues, and the published analysis; most of the rest of aging is cell-type specific.